<a href="https://colab.research.google.com/github/tougheye/Data_processing/blob/main/accreted_title_step_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The goal of the project is to take in TCS backend data then for each title -  of

1.   get the min, mid, max
2.   count the number of steps in the scale

This project is to support the step calculation of accreted dtitles

In [4]:
import pandas as pd

tcs_backend_folder = "/content/drive/MyDrive/Data/UCOP_Job_Codes_Summary"

In [5]:
tcs_backend_file = pd.ExcelFile(f'{tcs_backend_folder}/R-349 Job Codes Summary_07282026.xlsx')
tcs_backend_file.sheet_names
#

['Job Code Review',
 'Represented Job Codes',
 'Non-Represented Job Codes',
 'Shift and On Call Rates']

In [6]:
tcs_backend_tabs = tcs_backend_file.sheet_names
represented_job_codes_df = tcs_backend_file.parse('Represented Job Codes', skiprows=9, skipfooter=2)
represented_job_codes_df.shape

(128803, 30)

In [7]:
represented_active_df = represented_job_codes_df[represented_job_codes_df['Salary Grade Eff Status'] == 'A']
represented_active_df.shape

(109454, 30)

In [8]:
upte_represented_job_codes_df = represented_active_df[represented_active_df['Union Code'].isin(['HX', 'RX', 'TX'])]
upte_represented_job_codes_df.shape

(40459, 30)

In [9]:
upte_job_title_Eff_date_cnt = upte_represented_job_codes_df.groupby(['Salary Plan SETID','Job Code Description'])['Eff Date - Salary Grade'].nunique().reset_index(name='Date Count')
multi_eff_date_setid_jobCode_cnt = upte_job_title_Eff_date_cnt.where(upte_job_title_Eff_date_cnt['Date Count'] > 1).dropna()

In [10]:
# get the latest effective date for each job title
latest_eff_date_df = upte_represented_job_codes_df.groupby(['Salary Plan SETID','Job Code Description'])['Eff Date - Salary Grade'].max().reset_index(name='Latest Effective Date')

In [12]:
# filtered UPTE represented job codes
titles_not_updated = upte_represented_job_codes_df[upte_represented_job_codes_df['Eff Date - Salary Grade'] < '2026-07-01']
titles_not_updated.shape   # 1192 rows


(1192, 30)

In [13]:
# Merge to get the latest effective date for each job title for which the effective date is before July 1, 2026
# WILL KEEP THIS ONE TO LATER TAKE CARE OF
titles_not_updated_w_max_date = titles_not_updated.merge(latest_eff_date_df, on=['Salary Plan SETID', 'Job Code Description'], how='inner')

# Drop the LBNL business units as they are not supposed to receive the 5% ATB in July 2026
titles_not_updated_w_max_date = titles_not_updated_w_max_date[titles_not_updated_w_max_date['Salary Plan SETID'] != 'LBNL1']
titles_not_updated_w_max_date.shape       # 1130 rows

(1130, 31)

In [14]:

# Filter the rows with the latest effective date for each job title that have multiple effective dates
# THIS DF WILL BE CONCATENATED LATER TO CREATE THE FINAL DF WITH THE LATEST EFFECTIVE DATE

upte_represented_max_eff_date_df = upte_represented_job_codes_df\
    .merge(multi_eff_date_setid_jobCode_cnt,
           on=['Salary Plan SETID', 'Job Code Description'],
           how='inner')\
    .sort_values('Eff Date - Salary Grade', ascending=False)\
    .drop_duplicates(['Salary Plan SETID', 'Job Code Description', 'UCPATH Step', 'UC  Half Step'],
                     keep='first')

upte_represented_max_eff_date_df.shape

(1566, 31)

In [15]:
# Create unique identifier both in the master UPTE represented df and multiple effective date df

represented_key = upte_represented_job_codes_df['Salary Plan SETID'] + ' - ' + upte_represented_job_codes_df['Union Code'] + ' - ' + upte_represented_job_codes_df['Job Code Description']
unmatched_key = upte_represented_max_eff_date_df['Salary Plan SETID'] + ' - ' + upte_represented_max_eff_date_df['Union Code'] + ' - ' + upte_represented_max_eff_date_df['Job Code Description']

# filtering out the multiple effective date rows from the UPTE represented df based on the unique identifier keys
upte_represented_one_date_df = upte_represented_job_codes_df[~represented_key.isin(unmatched_key)]

In [16]:
upte_represented_final_df = pd.concat([upte_represented_one_date_df, upte_represented_max_eff_date_df.drop('Date Count', axis=1)])
upte_represented_final_df.shape

(39392, 30)

In [17]:
cols_to_keep = ['Salary Plan SETID', 'Job Code', 'Job Code Description', 'Union Code', 'UCPATH Step', 'UC  Half Step', 'Hrly Rate',
                'Eff Date - Salary Grade', 'Salary Grade Eff Status']
#

In [18]:
upte_represented_job_codes_refined_df = upte_represented_final_df[cols_to_keep]


,Salary Plan SETID,Job Code,Job Code Description,Union Code,UCPATH Step,UC Half Step,Hrly Rate,Eff Date - Salary Grade,Salary Grade Eff Status
217,BKCMP,004031,LIFEGUARD,TX,1.0,1.0,27.04,2026-07-05,A
218,BKCMP,004031,LIFEGUARD,TX,2.0,1.5,27.58,2026-07-05,A
219,BKCMP,004031,LIFEGUARD,TX,7.0,4.0,30.44,2026-07-05,A
220,BKCMP,004031,LIFEGUARD,TX,4.0,2.5,28.70,2026-07-05,A
221,BKCMP,004031,LIFEGUARD,TX,5.0,3.0,29.26,2026-07-05,A


In [19]:
upte_represented_job_codes_refined_df.head(50)

,Salary Plan SETID,Job Code,Job Code Description,Union Code,UCPATH Step,UC Half Step,Hrly Rate,Eff Date - Salary Grade,Salary Grade Eff Status
217,BKCMP,004031,LIFEGUARD,TX,1.0,1.0,27.040000,2026-07-05,A
218,BKCMP,004031,LIFEGUARD,TX,2.0,1.5,27.580000,2026-07-05,A
219,BKCMP,004031,LIFEGUARD,TX,7.0,4.0,30.440000,2026-07-05,A
220,BKCMP,004031,LIFEGUARD,TX,4.0,2.5,28.700000,2026-07-05,A
221,BKCMP,004031,LIFEGUARD,TX,5.0,3.0,29.260000,2026-07-05,A
222,BKCMP,004031,LIFEGUARD,TX,9.0,5.0,31.680000,2026-07-05,A
223,BKCMP,004031,LIFEGUARD,TX,10.0,5.5,32.310000,2026-07-05,A
224,BKCMP,004031,LIFEGUARD,TX,6.0,3.5,29.850000,2026-07-05,A
225,BKCMP,004031,LIFEGUARD,TX,8.0,4.5,31.060000,2026-07-05,A
226,BKCMP,004031,LIFEGUARD,TX,3.0,2.0,28.130000,2026-07-05,A


Codes above are all from the [TCS payscale notebook](https://colab.research.google.com/drive/1MwFYF32tmy4eGs_cS6p0rc_hfDQ3drTM?usp=chrome_ntp#scrollTo=H9473xBkUotq)

In [31]:
upte_represented_job_codes_refined_df[(upte_represented_job_codes_refined_df['Salary Plan SETID'] == 'BKCMP') & (upte_represented_job_codes_refined_df['Job Code Description'] == 'PSYCHOLOGIST 1')]

,Salary Plan SETID,Job Code,Job Code Description,Union Code,UCPATH Step,UC Half Step,Hrly Rate,Eff Date - Salary Grade,Salary Grade Eff Status
4352,BKCMP,009384,PSYCHOLOGIST 1,HX,21.0,21.0,58.317505,2026-07-01,A
4353,BKCMP,009384,PSYCHOLOGIST 1,HX,25.0,25.0,63.124200,2026-07-01,A
4354,BKCMP,009384,PSYCHOLOGIST 1,HX,15.0,15.0,51.787965,2026-07-01,A
4355,BKCMP,009384,PSYCHOLOGIST 1,HX,8.0,8.0,45.085340,2026-07-01,A
4356,BKCMP,009384,PSYCHOLOGIST 1,HX,18.0,18.0,54.966197,2026-07-01,A
4357,BKCMP,009384,PSYCHOLOGIST 1,HX,28.0,28.0,66.994727,2026-07-01,A
4358,BKCMP,009384,PSYCHOLOGIST 1,HX,2.0,2.0,40.042639,2026-07-01,A
4359,BKCMP,009384,PSYCHOLOGIST 1,HX,5.0,5.0,42.489253,2026-07-01,A
4360,BKCMP,009384,PSYCHOLOGIST 1,HX,9.0,9.0,45.990029,2026-07-01,A
4361,BKCMP,009384,PSYCHOLOGIST 1,HX,23.0,23.0,60.677586,2026-07-01,A


In [35]:
import math
import numpy as np
business_unit = 'BKCMP'

job_details_dict = {}
  #define the current business unit
current_bus_unit = upte_represented_job_codes_refined_df[
      upte_represented_job_codes_refined_df['Salary Plan SETID'] == business_unit]

# loop through the bargaining units
for bargaining_unit in current_bus_unit['Union Code'].unique():
  job_details_dict[bargaining_unit] = {}

  # define the current bargaining unit
  current_bus_unit_bu_df = current_bus_unit[current_bus_unit['Union Code'] == bargaining_unit]

  title_list = sorted(current_bus_unit_bu_df['Job Code Description'].unique())

  for title in title_list:
    current_title_df = current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]

    # Filter out NaN values from 'UC Half Step' to get valid steps
    valid_half_steps = [float(k) for k in current_title_df['UC  Half Step'].unique() if not math.isnan(k)]

    half_step_list = sorted(valid_half_steps)

    # Initialize Mid Step and Mid Step Rate to NaN
    mid_step_value = np.nan
    mid_step_rate = np.nan
    total_steps = len(half_step_list)

    if half_step_list: # Only calculate mid step if there are actual steps
        mid_step_index = len(half_step_list) // 2
        mid_step_value = half_step_list[mid_step_index]

        # Find the row in current_title_df that matches this mid_step_value
        mid_step_rate_series = current_title_df[current_title_df['UC  Half Step'] == mid_step_value]['Hrly Rate']

        if not mid_step_rate_series.empty:
            mid_step_rate = mid_step_rate_series.iloc[0]

    job_details_dict[bargaining_unit][title] = {
          'Title' : title,
          'Job Code' : current_title_df['Job Code'].unique()[0],
          'Min Rate' : current_title_df['Hrly Rate'].min(),
          'Max Rate' : current_title_df['Hrly Rate'].max(),
          'Total Steps' : total_steps,
          'TCS Effective Date': current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]['Eff Date - Salary Grade'].unique().strftime("%m/%d/%Y"),
          'Mid Step': mid_step_value,
          'Mid Step Rate': mid_step_rate
    }

In [44]:
import math
# dictionary to store the job details
job_details_dict = {}

business_unit_list = upte_represented_job_codes_refined_df['Salary Plan SETID'].unique()

for business_unit in business_unit_list:
  job_details_dict[business_unit] = {}
  #define the current business unit
  current_bus_unit = upte_represented_job_codes_refined_df[
      upte_represented_job_codes_refined_df['Salary Plan SETID'] == business_unit]

  # loop through the bargaining units
  for bargaining_unit in current_bus_unit['Union Code'].unique():
    job_details_dict[business_unit][bargaining_unit] = {}

    # define the current bargaining unit
    current_bus_unit_bu_df = current_bus_unit[current_bus_unit['Union Code'] == bargaining_unit]

    title_list = sorted(current_bus_unit_bu_df['Job Code Description'].unique())

    for title in title_list:
      current_title_df = current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]

      # Filter out NaN values from 'UC Half Step' to get valid steps
      valid_half_steps = [float(k) for k in current_title_df['UC  Half Step'].unique() if not math.isnan(k)]

      half_step_list = sorted(valid_half_steps)

      # Initialize Mid Step and Mid Step Rate to NaN
      mid_step_value = np.nan
      mid_step_rate = np.nan
      total_steps = len(half_step_list)

      if half_step_list: # Only calculate mid step if there are actual steps
          mid_step_index = len(half_step_list) // 2
          mid_step_value = half_step_list[mid_step_index]

          # Find the row in current_title_df that matches this mid_step_value
          mid_step_rate_series = current_title_df[current_title_df['UC  Half Step'] == mid_step_value]['Hrly Rate']

          if not mid_step_rate_series.empty:
              mid_step_rate = mid_step_rate_series.iloc[0]

      job_details_dict[business_unit][bargaining_unit][title] = {
            'Title' : title,
            'Job Code' : current_title_df['Job Code'].unique()[0],
            'Min Rate' : current_title_df['Hrly Rate'].min(),
            'Max Rate' : current_title_df['Hrly Rate'].max(),
            'Total Steps' : total_steps,
            'TCS Effective Date': current_bus_unit_bu_df[current_bus_unit_bu_df['Job Code Description'] == title]['Eff Date - Salary Grade'].unique().strftime("%m/%d/%Y"),
            'Mid Step': mid_step_value,
            'Mid Step Rate': mid_step_rate
      }



In [47]:
job_details_dict['LAMED']['TX']['READER FOR THE BLIND']

{'Title': 'READER FOR THE BLIND',
 'Job Code': '006677',
 'Min Rate': 26.25,
 'Max Rate': 26.25,
 'Total Steps': 1,
 'TCS Effective Date': array(['07/05/2026'], dtype=object),
 'Mid Step': 1.0,
 'Mid Step Rate': np.float64(26.25)}

In [51]:
all_dfs = {}
for business_unit, bargaining_units in job_details_dict.items():
  for bargaining_unit, titles in bargaining_units.items():
    df_name = f"{business_unit}_{bargaining_unit}_df"
    all_dfs[df_name] = pd.DataFrame(titles).T.drop

In [52]:
all_dfs['BKCMP_HX_df'].head()

,Title,Job Code,Min Rate,Max Rate,Total Steps,TCS Effective Date,Mid Step,Mid Step Rate
ATH TRAINER 1 HX,ATH TRAINER 1 HX,005304,NaN,NaN,0,[07/01/2026],NaN,NaN
ATH TRAINER 2 HX,ATH TRAINER 2 HX,005305,NaN,NaN,0,[07/01/2026],NaN,NaN
ATH TRAINER 3 HX,ATH TRAINER 3 HX,005306,NaN,NaN,0,[07/01/2026],NaN,NaN
ATH TRAINER 4 HX,ATH TRAINER 4 HX,005307,NaN,NaN,0,[07/01/2026],NaN,NaN
BEH HEALTH COUNSELOR 2 HX,BEH HEALTH COUNSELOR 2 HX,004458,54.62,67.9,12,[07/05/2026],7.0,61.51
